# Energy forecasting and anomaly detection — revised evaluation

Run cells from top to bottom in Google Colab. This notebook reads `energy_project/synthetic_energy_data.csv` and writes new files to `energy_project/revised_outputs/`, leaving the earlier outputs intact. It rebuilds features, applies chronological target-aware splits, selects models and anomaly thresholds on validation, and evaluates the test period once. Data are synthetic; performance here does not establish performance on real homes. The `1day` target is an **instantaneous reading 24 hours ahead**, not total daily energy.

In [ ]:
from pathlib import Path
import os, json, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_recall_fscore_support, confusion_matrix
from sklearn.impute import SimpleImputer
import sklearn

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/energy_project')
except ImportError:
    ROOT = Path.cwd()

DATA = ROOT / 'synthetic_energy_data.csv'
OUT = ROOT / 'revised_outputs'
OUT.mkdir(parents=True, exist_ok=True)
assert DATA.exists(), f'Missing {DATA}; set ROOT to the folder containing synthetic_energy_data.csv'
raw = pd.read_csv(DATA, parse_dates=['date']).sort_values('date').reset_index(drop=True)
assert raw.date.is_unique and raw.date.notna().all()
assert {'Appliances','anomaly_flag','anomaly_type'}.issubset(raw.columns)
print('Source rows:', len(raw), 'date range:', raw.date.min(), 'to', raw.date.max())
print('Output:', OUT)


## 1. Features, targets and chronological boundaries

Use only measurements available at the forecast origin. Past rolling features exclude the current row; the current appliance reading is itself observable at the origin. Target timestamps are checked explicitly. Rows whose labels cross a split boundary are purged from the earlier split.

In [ ]:
df = raw.copy()
dt = df.date
df['hour_of_day'] = dt.dt.hour + dt.dt.minute / 60
df['sin_hour'] = np.sin(2*np.pi*df.hour_of_day/24)
df['cos_hour'] = np.cos(2*np.pi*df.hour_of_day/24)
df['dow'] = dt.dt.dayofweek
df['sin_dow'] = np.sin(2*np.pi*df.dow/7)
df['cos_dow'] = np.cos(2*np.pi*df.dow/7)
df['sin_month'] = np.sin(2*np.pi*dt.dt.month/12)
df['cos_month'] = np.cos(2*np.pi*dt.dt.month/12)
df['is_weekend'] = (df.dow >= 5).astype(int)
for label, steps in {'10min':1,'1h':6,'6h':36,'1day':144,'1week':1008}.items():
    df[f'Appliances_lag_{label}'] = df.Appliances.shift(steps)
for label, steps in {'1h':6,'6h':36,'24h':144}.items():
    past = df.Appliances.shift(1).rolling(steps, min_periods=steps)
    df[f'Appliances_rollmean_{label}'] = past.mean()
    df[f'Appliances_rollstd_{label}'] = past.std()
df['T_out_lag_1h'] = df.T_out.shift(6)
for h, steps in {'1h':6,'1day':144}.items():
    df[f'target_{h}'] = df.Appliances.shift(-steps)
    df[f'target_time_{h}'] = df.date.shift(-steps)
    expected = df.date + pd.Timedelta(minutes=10*steps)
    valid = df[f'target_time_{h}'].eq(expected)
    df.loc[~valid, [f'target_{h}', f'target_time_{h}']] = np.nan

# Retain full feature history until all lags and targets have been computed.
df = df.iloc[1008:-144].copy().reset_index(drop=True)
assert df[['target_1h','target_1day']].notna().all().all(), 'Unexpected time gaps or missing targets; inspect data frequency.'
n = len(df)
val_start = df.date.iloc[int(.70*n)]
test_start = df.date.iloc[int(.85*n)]
origins = {'train':df.date < val_start,
           'val':(df.date >= val_start)&(df.date < test_start),
           'test':df.date >= test_start}

# Explicit allowlist: current measured sensors/weather and causal historical load,
# plus calendar values known in advance. No ground-truth anomaly label or future target.
sensor_cols = [c for c in raw.columns if c in
    ['Appliances','lights','T_out','Press_mm_hg','RH_out','Windspeed','Visibility',
     'Tdewpoint','Solar_Radiation','Occupancy_Count','rv1','rv2'] or
    (c.startswith(('T','RH')) and c[1:].isdigit())]
derived_cols = [c for c in df.columns if c.startswith(('Appliances_lag_','Appliances_roll','T_out_lag_'))]
calendar_cols = ['hour_of_day','sin_hour','cos_hour','dow','sin_dow','cos_dow','sin_month','cos_month','is_weekend']
FEATURES = list(dict.fromkeys(sensor_cols + derived_cols + calendar_cols))
assert not set(FEATURES) & {'anomaly_flag','anomaly_type','target_1h','target_1day','date'}
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in FEATURES)
print('Split origins:', val_start, test_start, '| features:', len(FEATURES))
print('Note: weather and occupancy columns are contemporaneous measured values. For deployment, remove those not available when issuing the forecast or replace them with forecasts.')


## 2. Forecast validation, selection and final test

Selection uses validation RMSE. Every model and baseline is scored on the same timestamped rows within a horizon. The seasonal baseline uses the observation exactly one week before the **target timestamp**. The old MAPE used a floor of 10 in its denominator; WAPE is reported here and the old metric is not silently carried forward.

In [ ]:
try:
    import xgboost as xgb
    HAVE_XGB = True
except ImportError:
    if 'google.colab' in sys.modules:
        import subprocess
        subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
        import xgboost as xgb
        HAVE_XGB = True
    else:
        HAVE_XGB = False
        print('XGBoost unavailable in this runtime; install it to include that model.')

def make_model(name):
    if name == 'linear': return LinearRegression()
    if name == 'rf': return RandomForestRegressor(n_estimators=80, max_depth=10, n_jobs=-1, random_state=42)
    if name == 'xgb': return xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=.05,
                                             objective='reg:squarederror', n_jobs=-1, random_state=42)
    raise ValueError(name)

def forecast_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    e = np.abs(y-p)
    return {'n':len(y),'RMSE':float(np.sqrt(np.mean((y-p)**2))),
            'MAE':float(e.mean()), 'WAPE_pct':float(100*e.sum()/np.abs(y).sum())}

models = ['linear','rf'] + (['xgb'] if HAVE_XGB else [])
forecast_rows, prediction_frames, selected_models = [], [], {}
lookup = raw.set_index('date').Appliances
for horizon in ['1h','1day']:
    target = f'target_{horizon}'
    target_time = f'target_time_{horizon}'
    # Eliminate labels that extend beyond the training/validation boundary.
    tr = df[origins['train'] & (df[target_time] < val_start)].copy()
    va = df[origins['val'] & (df[target_time] < test_start)].copy()
    te = df[origins['test']].copy()
    for split_name, part in [('train',tr),('val',va),('test',te)]:
        assert (part[target_time] > part.date).all()
        if split_name == 'train': assert (part[target_time] < val_start).all()
        if split_name == 'val': assert (part[target_time] < test_start).all()
    print(horizon, 'train/val/test:', len(tr), len(va), len(te))

    def baselines(part):
        naive = part.Appliances.to_numpy(float)
        seasonal_dates = part[target_time] - pd.Timedelta(days=7)
        seasonal = lookup.reindex(seasonal_dates).to_numpy(float)
        assert np.isfinite(seasonal).all(), 'Missing seasonal baseline timestamps'
        return {'naive':naive,'seasonal_naive':seasonal}

    val_predictions = baselines(va)
    fitted = {}
    for name in models:
        model = make_model(name)
        model.fit(tr[FEATURES], tr[target])
        fitted[name] = model
        val_predictions[name] = model.predict(va[FEATURES])
    for name, pred in val_predictions.items():
        forecast_rows.append({'split':'val','horizon':horizon,'model':name,
                              **forecast_metrics(va[target],pred)})
    best = min(models, key=lambda name: forecast_metrics(va[target],val_predictions[name])['RMSE'])
    print('Selected by validation RMSE:', best)
    selected_models[horizon] = best
    prediction_frames.append(pd.DataFrame({'origin_time':va.date,'target_time':va[target_time],
       'horizon':horizon,'split':'val','actual':va[target],
       **{f'pred_{name}':pred for name,pred in val_predictions.items()}}))

    # Refit the selected model using all training and validation rows whose
    # labels are known before the test period starts. Hold out test until now.
    fit_rows = df[(df.date < test_start)&(df[target_time] < test_start)]
    final_model = make_model(best)
    final_model.fit(fit_rows[FEATURES], fit_rows[target])
    if horizon == '1h': hourly_model = final_model
    test_predictions = baselines(te)
    test_predictions[best] = final_model.predict(te[FEATURES])
    for name,pred in test_predictions.items():
        forecast_rows.append({'split':'test','horizon':horizon,'model':name,
                              **forecast_metrics(te[target],pred)})
    prediction_frames.append(pd.DataFrame({'origin_time':te.date,'target_time':te[target_time],
       'horizon':horizon,'split':'test','actual':te[target],
       **{f'pred_{name}':pred for name,pred in test_predictions.items()}}))

forecast_results = pd.DataFrame(forecast_rows)
forecast_predictions = pd.concat(prediction_frames,ignore_index=True)
forecast_results.to_csv(OUT/'model_results.csv',index=False)
forecast_predictions.to_csv(OUT/'forecast_predictions.csv',index=False)
display(forecast_results.round(2))


## 3. Peak, time and anomaly slices

The selected model is tested on peaks and ordinary readings separately. These summaries describe model errors within the synthetic test period.

In [ ]:
slices = []
for horizon in ['1h','1day']:
    p = forecast_predictions.query('split == "test" and horizon == @horizon').copy()
    p['origin_time'] = pd.to_datetime(p.origin_time)
    p['hour'] = p.origin_time.dt.hour
    p['month'] = p.origin_time.dt.month
    p['anomaly'] = df.set_index('date').anomaly_flag.reindex(p.origin_time).to_numpy()
    cutoff = raw.loc[raw.date < val_start,'Appliances'].quantile(.90)
    p['load_band'] = np.where(p.actual >= cutoff, 'peak >= train P90', 'below train P90')
    col = f'pred_{selected_models[horizon]}'
    for dimension in ['load_band','hour','month','anomaly']:
        for group,g in p.groupby(dimension):
            slices.append({'horizon':horizon,'dimension':dimension,'group':str(group),
                           **forecast_metrics(g.actual,g[col])})
slice_results = pd.DataFrame(slices)
slice_results.to_csv(OUT/'forecast_slices.csv',index=False)
display(slice_results.query('dimension == "load_band"').round(2))


## 4. Anomaly detection

Fit Isolation Forest on the train period only. Tune thresholds and combinations on validation only. The zero reading rule is specific to this synthetic generator, whose normal appliance readings have a positive floor. Exclude the generator's clipped temperature bounds from the stuck rule; this choice needs review on real data.

In [ ]:
anom = raw[['date','Appliances','anomaly_flag','anomaly_type']+[f'T{i}' for i in range(1,10)]].copy()
anom['split'] = np.where(anom.date < val_start,'train',np.where(anom.date < test_start,'val','test'))
past = anom.Appliances.shift(1).rolling(144,min_periods=24)
anom['zscore'] = ((anom.Appliances-past.mean())/past.std()).replace([np.inf,-np.inf],np.nan).fillna(0)
anom['drop'] = anom.Appliances.eq(0).astype(int)

# Use deviations from a trailing identical run. Six ten-minute readings are
# sufficient for the shortest candidate. A clipped floor can stay flat normally.
for length in [6,9,12]:
    flags = []
    for col in [f'T{i}' for i in range(1,10)]:
        r = anom[col].rolling(length,min_periods=length)
        flags.append(((r.max()-r.min()).abs() < 1e-8) &
                     anom[col].gt(10.05) & anom[col].lt(31.95))
    anom[f'stuck_{length}'] = np.logical_or.reduce(flags).astype(int)

iso_cols = ['Appliances']+[f'T{i}' for i in range(1,10)]
train_mask = anom.split.eq('train')
imp = SimpleImputer(strategy='median').fit(anom.loc[train_mask,iso_cols])
X_iso_train = imp.transform(anom.loc[train_mask,iso_cols])
X_iso = imp.transform(anom[iso_cols])
iso = IsolationForest(n_estimators=100,max_samples=256,random_state=42,n_jobs=-1).fit(X_iso_train)
scores = -iso.score_samples(X_iso)
train_scores = scores[train_mask.to_numpy()]

def counts(y,p):
    tn,fp,fn,tp = confusion_matrix(y,p,labels=[0,1]).ravel()
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return dict(n=len(y),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),
                precision=precision,recall=recall,f1=f1,alert_rate=(tp+fp)/len(y))

candidate = {'drop':anom['drop'].to_numpy(int)}
for z in [3.0,3.5,4.0,5.0]:
    candidate[f'z_{z}'] = anom.zscore.abs().ge(z).to_numpy(int)
for length in [6,9,12]:
    candidate[f'stuck_{length}'] = anom[f'stuck_{length}'].to_numpy(int)
for tail in [.005,.01,.02]:
    threshold = np.quantile(train_scores,1-tail)
    candidate[f'iso_{tail}'] = (scores > threshold).astype(int)

val_mask = anom.split.eq('val').to_numpy()
y_val = anom.loc[val_mask,'anomaly_flag'].to_numpy(int)
chosen = {}
for family,prefix in [('z','z_'),('stuck','stuck_'),('iso','iso_')]:
    options = [k for k in candidate if k.startswith(prefix)]
    chosen[family] = max(options,key=lambda k:(counts(y_val,candidate[k][val_mask])['f1'],
                                                 counts(y_val,candidate[k][val_mask])['precision']))
chosen['drop'] = 'drop'
print('Validation choices:',chosen)

from itertools import combinations
families = list(chosen)
combos = [group for size in range(1,len(families)+1) for group in combinations(families,size)]
def combo_pred(group):
    return np.logical_or.reduce([candidate[chosen[f]] for f in group]).astype(int)
selected_combo = max(combos,key=lambda g:(counts(y_val,combo_pred(g)[val_mask])['f1'],
                                           counts(y_val,combo_pred(g)[val_mask])['precision']))
print('Selected combination:',selected_combo)

for family,k in chosen.items():
    anom[f'pred_anomaly_{family}'] = candidate[k]
anom['pred_anomaly_combined'] = combo_pred(selected_combo)
anom['pred_anomaly_zscore'] = anom['pred_anomaly_z']
methods = ['z','stuck','iso','drop','combined']
anomaly_rows=[]
for split in ['val','test']:
    part = anom[anom.split.eq(split)]
    for method in methods:
        anomaly_rows.append({'split':split,'method':method,
                             **counts(part.anomaly_flag,part[f'pred_anomaly_{method}'])})
anomaly_results = pd.DataFrame(anomaly_rows)
anomaly_results.to_csv(OUT/'anomaly_results.csv',index=False)
anom.to_csv(OUT/'anomaly_predictions.csv',index=False)
display(anomaly_results.round(4))

# Verify that summary counts are derived from precisely the rows just exported.
check = pd.read_csv(OUT/'anomaly_predictions.csv')
for row in anomaly_rows:
    part = check[check.split.eq(row['split'])]
    assert all(counts(part.anomaly_flag,part[f"pred_anomaly_{row['method']}"])[k] == row[k]
               for k in ['TP','FP','FN','TN'])


## 5. Event detection and false alerts

An event is a contiguous block of anomalous ten-minute rows of one type. It is detected if at least one row in that block is flagged. A false alert is the first flagged normal row in a contiguous flagged run; count those by test day.

In [ ]:
test_anom = anom[anom.split.eq('test')].copy()
test_anom['event_start'] = test_anom.anomaly_flag.eq(1) & (
    test_anom.anomaly_flag.shift(fill_value=0).eq(0) |
    test_anom.anomaly_type.ne(test_anom.anomaly_type.shift()))
test_anom['event_id'] = test_anom.event_start.cumsum()
events = test_anom[test_anom.anomaly_flag.eq(1)].groupby(['event_id','anomaly_type'],as_index=False).agg(
    detected=('pred_anomaly_combined','max'), start=('date','min'), duration_rows=('date','size'))
event_summary = events.groupby('anomaly_type').agg(events=('detected','size'),caught=('detected','sum'))
event_summary['event_recall'] = event_summary.caught/event_summary.events
event_summary.to_csv(OUT/'anomaly_events.csv')
normal_alert = test_anom.pred_anomaly_combined.eq(1) & test_anom.anomaly_flag.eq(0)
false_starts = normal_alert & ~normal_alert.shift(fill_value=False)
false_per_day = false_starts.groupby(test_anom.date.dt.date).sum()
print('False alert episodes per test day: mean',round(false_per_day.mean(),2),
      'median',false_per_day.median(),'total',false_starts.sum())
display(event_summary.round(3))


## 6. Charts and optional SHAP explanation

In [ ]:
p = forecast_predictions.query('split == "test" and horizon == "1h"').copy()
p['target_time'] = pd.to_datetime(p.target_time)
best = selected_models['1h']
p['error'] = p.actual-p[f'pred_{best}']
peak = p.loc[p.actual.idxmax(),'target_time']
window = p[p.target_time.between(peak-pd.Timedelta(hours=6),peak+pd.Timedelta(hours=6))]
fig,axes=plt.subplots(2,1,figsize=(13,7),sharex=True)
axes[0].plot(window.target_time,window.actual,label='Actual',lw=1.2)
axes[0].plot(window.target_time,window[f'pred_{best}'],label=f'{best} 1h ahead',lw=1.2)
axes[0].set_ylabel('Appliances (dataset units)');axes[0].legend()
axes[1].plot(window.target_time,window.error,color='firebrick');axes[1].axhline(0,color='black',lw=.7)
axes[1].set_ylabel('Actual − predicted');axes[1].set_xlabel('Target time')
fig.suptitle(f'Test period around maximum observed load; {best}, 1h ahead')
fig.tight_layout();fig.savefig(OUT/'pred_vs_actual.png',dpi=150);plt.show()

t = test_anom.copy()
is_fp = t.pred_anomaly_combined.eq(1) & t.anomaly_flag.eq(0)
worst_day = is_fp.groupby(t.date.dt.date).sum().idxmax()
w = t[t.date.dt.date.eq(worst_day)]
fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=True)
axes[0].plot(w.date,w.Appliances,lw=.8,color='steelblue',label='Appliances')
for label,m,color in [('Caught',w.anomaly_flag.eq(1)&w.pred_anomaly_combined.eq(1),'green'),
                      ('Missed',w.anomaly_flag.eq(1)&w.pred_anomaly_combined.eq(0),'red'),
                      ('False positive',w.anomaly_flag.eq(0)&w.pred_anomaly_combined.eq(1),'orange')]:
    axes[0].scatter(w.loc[m,'date'],w.loc[m,'Appliances'],s=18,c=color,label=label,zorder=4)
axes[0].set_ylabel('Appliances (dataset units)');axes[0].legend()
for col in ['T6','T7','T9']: axes[1].plot(w.date,w[col],lw=.8,label=col)
axes[1].set_ylabel('Sensor temperature');axes[1].legend();axes[1].set_xlabel('Time')
fig.suptitle(f'Held-out test anomalies and false alerts: {worst_day}')
fig.tight_layout();fig.savefig(OUT/'anomaly_detection.png',dpi=150);plt.show()

if best in ['rf','xgb']:
    try:
        try:
            import shap
        except ImportError:
            if 'google.colab' in sys.modules:
                import subprocess
                subprocess.check_call([sys.executable,'-m','pip','install','-q','shap'])
                import shap
            else:
                raise
        sample = df.loc[origins['test'],FEATURES].sample(n=min(200,len(df.loc[origins['test']])),random_state=42)
        explainer = shap.TreeExplainer(hourly_model)
        values = explainer.shap_values(sample)
        shap.summary_plot(values,sample,show=False)
        plt.title(f'{best}: 1h ahead, held-out test sample n={len(sample)}')
        plt.tight_layout();plt.savefig(OUT/'shap_summary.png',dpi=150,bbox_inches='tight');plt.show()
    except Exception as exc:
        print('SHAP plot unavailable in this runtime:',type(exc).__name__,str(exc)[:250])
else:
    print('Selected hourly model is linear; tree SHAP summary omitted.')


## 7. Run record and limitations

In [ ]:
manifest = {'source':str(DATA),'source_rows':len(raw),'train_origin_end':str(val_start),
 'test_origin_start':str(test_start),'selected_forecast_models':selected_models,
 'selected_anomaly_thresholds':chosen,'selected_anomaly_components':list(selected_combo),
 'features':FEATURES,'seed':42,'python':platform.python_version(),
 'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__,
 'xgboost':getattr(xgb,'__version__',None) if HAVE_XGB else None,
 'limitations':['Synthetic source; real-home validity untested',
                'Contemporaneous weather/occupancy require availability at forecast origin',
                'Zero and clip-bound anomaly rules depend on this generator',
                'No calibrated prediction intervals; add only after a separate calibration period']}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Completed. Results saved to',OUT)
